In [8]:
import os
import time
import numpy as np
import torch
import segmentation_models_pytorch as smp
import onnxruntime as ort
import matplotlib.pyplot as plt

def get_model(model_name, device):
    if model_name == "DeepLabV3":
        model = smp.DeepLabV3(encoder_name="mobileone_s0", encoder_weights=None, in_channels=3, classes=1)
    elif model_name == "UnetPlusPlus":
        model = smp.UnetPlusPlus(encoder_name="mobileone_s0", encoder_weights=None, in_channels=3, classes=1)
    elif model_name == "Segformer":
        model = smp.Segformer(encoder_name="mit_b0", encoder_weights=None, in_channels=3, classes=1)
    else:
        raise ValueError(f"Unknown model: {model_name}")
    return model.to(device).eval()

def get_expected_memory_mb(model):
    param_size = sum(p.numel() * p.element_size() for p in model.parameters())
    buffer_size = sum(b.numel() * b.element_size() for b in model.buffers())
    return (param_size + buffer_size) / (1024 * 1024)

def export_to_onnx(model, dummy_input, save_path):
    torch.onnx.export(
        model,
        dummy_input,
        save_path,
        export_params=True,
        opset_version=11,
        do_constant_folding=True,
        input_names=['input'],
        output_names=['output']
    )

def benchmark_cpu_raw(model, dummy_input, runs):
    for _ in range(10):
        with torch.no_grad():
            _ = model(dummy_input)
            
    times = []
    for _ in range(runs):
        start_time = time.perf_counter()
        with torch.no_grad():
            _ = model(dummy_input)
        times.append((time.perf_counter() - start_time) * 1000)
        
    return times

def benchmark_gpu_raw(model, dummy_input, runs):
    for _ in range(10):
        with torch.no_grad():
            _ = model(dummy_input)
    torch.cuda.synchronize()
    
    times = []
    start_events = [torch.cuda.Event(enable_timing=True) for _ in range(runs)]
    end_events = [torch.cuda.Event(enable_timing=True) for _ in range(runs)]
    
    for i in range(runs):
        start_events[i].record()
        with torch.no_grad():
            _ = model(dummy_input)
        end_events[i].record()
        
    torch.cuda.synchronize()
    
    for i in range(runs):
        times.append(start_events[i].elapsed_time(end_events[i]))
        
    return times

def benchmark_onnx_raw(onnx_path, dummy_input, device_name, runs):
    providers = ['CUDAExecutionProvider', 'CPUExecutionProvider'] if device_name == "cuda" else ['CPUExecutionProvider']
    
    session_options = ort.SessionOptions()
    session_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    
    session = ort.InferenceSession(onnx_path, sess_options=session_options, providers=providers)
    input_name = session.get_inputs()[0].name
    output_name = session.get_outputs()[0].name
    
    if device_name == "cuda":
        io_binding = session.io_binding()
        io_binding.bind_input(
            name=input_name,
            device_type='cuda',
            device_id=0,
            element_type=np.float32,
            shape=tuple(dummy_input.shape),
            buffer_ptr=dummy_input.data_ptr()
        )
        io_binding.bind_output(output_name, 'cuda')
        
        for _ in range(10):
            session.run_with_iobinding(io_binding)
            
        times = []
        for _ in range(runs):
            start_time = time.perf_counter()
            session.run_with_iobinding(io_binding)
            torch.cuda.synchronize()
            times.append((time.perf_counter() - start_time) * 1000)
            
        return times
        
    else:
        dummy_input_np = dummy_input.cpu().numpy()
        for _ in range(10):
            _ = session.run(None, {input_name: dummy_input_np})
            
        times = []
        for _ in range(runs):
            start_time = time.perf_counter()
            _ = session.run(None, {input_name: dummy_input_np})
            times.append((time.perf_counter() - start_time) * 1000)
            
        return times

def run_benchmarks_and_chart():
    if "CUDAExecutionProvider" not in ort.get_available_providers():
        print("Missing onnxruntime-gpu package. ONNX will not run with CUDA.")
        
    models = ["DeepLabV3", "UnetPlusPlus", "Segformer"]
    devices = ["cpu"]
    if torch.cuda.is_available():
        devices.append("cuda")
        
    input_size = (1, 3, 512, 512)
    runs = 3
    
    print(f"{'Device':<8} | {'Format':<10} | {'Model':<15} | {'Time (ms)':<15} | {'Expected Mem (MB)'}")
    print("-" * 75)
    
    chart_data_times = {}

    for device_name in devices:
        dummy_input_pt = torch.randn(input_size, device=device_name, dtype=torch.float32)
        
        for model_name in models:
            config_key = f"{device_name.upper()}_{model_name}"
            
            model = get_model(model_name, device_name)
            expected_mem = get_expected_memory_mb(model)
            
            if device_name == "cuda":
                raw_times = benchmark_gpu_raw(model, dummy_input_pt, runs)
            else:
                raw_times = benchmark_cpu_raw(model, dummy_input_pt, runs)
                
            chart_data_times[f"PyTorch baseline_{config_key}"] = raw_times
            
            time_str = f"{np.mean(raw_times):.2f} ± {np.std(raw_times):.2f}"
            print(f"{device_name:<8} | {'PyTorch':<10} | {model_name:<15} | {time_str:<15} | {expected_mem:.2f}")
            
            onnx_path = f"{model_name}.onnx"
            if not os.path.exists(onnx_path):
                export_model = get_model(model_name, "cpu")
                export_input = torch.randn(input_size, device="cpu", dtype=torch.float32)
                export_to_onnx(export_model, export_input, onnx_path)
                del export_model
                
            onnx_raw_times = benchmark_onnx_raw(onnx_path, dummy_input_pt, device_name, runs)
            
            chart_data_times[f"ONNX optimized_{config_key}"] = onnx_raw_times
            
            onnx_time_str = f"{np.mean(onnx_raw_times):.2f} ± {np.std(onnx_raw_times):.2f}"
            print(f"{device_name:<8} | {'ONNX':<10} | {model_name:<15} | {onnx_time_str:<15} | {expected_mem:.2f}")
            
            del model
            if device_name == "cuda":
                torch.cuda.empty_cache()

    print("\nGenerating charts...")
    
    output_dir = "benchmark_charts"
    os.makedirs(output_dir, exist_ok=True)
    
    for model in models:
        labels = []
        data = []
        for config, times in chart_data_times.items():
            if model in config:
                label = config.replace(f"_{model}", "")
                labels.append(label)
                data.append(times)

        plt.figure(figsize=(10, 6))
        plt.boxplot(data, labels=labels, patch_artist=True, 
                    boxprops=dict(facecolor='lightblue', color='blue'),
                    medianprops=dict(color='red'))
        
        plt.title(f"{model} inferencijos laiko pasiskirstymas")
        plt.ylabel("Laikas (ms)")
        plt.grid(axis='y', linestyle='--', alpha=0.7)
        plt.xticks(rotation=15)
        plt.tight_layout()
        
        save_path = os.path.join(output_dir, f"boxplot_{model}.png")
        plt.savefig(save_path)
        plt.close()

    print(f"All charts saved in directory: '{output_dir}/'")

run_benchmarks_and_chart()

Device   | Format     | Model           | Time (ms)       | Expected Mem (MB)
---------------------------------------------------------------------------
cpu      | PyTorch    | DeepLabV3       | 236.28 ± 4.12   | 49.15
cpu      | ONNX       | DeepLabV3       | 213.93 ± 1.36   | 49.15
cpu      | PyTorch    | UnetPlusPlus    | 306.73 ± 14.81  | 38.06
cpu      | ONNX       | UnetPlusPlus    | 182.47 ± 0.29   | 38.06
cpu      | PyTorch    | Segformer       | 121.38 ± 1.59   | 14.17
cpu      | ONNX       | Segformer       | 63.51 ± 0.61    | 14.17
cuda     | PyTorch    | DeepLabV3       | 24.20 ± 0.03    | 49.15
cuda     | ONNX       | DeepLabV3       | 19.50 ± 0.09    | 49.15
cuda     | PyTorch    | UnetPlusPlus    | 23.38 ± 1.10    | 38.06
cuda     | ONNX       | UnetPlusPlus    | 22.75 ± 0.05    | 38.06
cuda     | PyTorch    | Segformer       | 7.30 ± 0.12     | 14.17
cuda     | ONNX       | Segformer       | 6.52 ± 0.03     | 14.17

Generating charts...


/tmp/nix-shell.fRjzOZ/ipykernel_684607/1975263801.py:190: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(data, labels=labels, patch_artist=True,
/tmp/nix-shell.fRjzOZ/ipykernel_684607/1975263801.py:190: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(data, labels=labels, patch_artist=True,
/tmp/nix-shell.fRjzOZ/ipykernel_684607/1975263801.py:190: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(data, labels=labels, patch_artist=True,


All charts saved in directory: 'benchmark_charts/'


In [11]:
import os
import time
import numpy as np
import torch
import segmentation_models_pytorch as smp
import onnxruntime as ort
import matplotlib.pyplot as plt

def get_model(model_name, device):
    if model_name == "DeepLabV3":
        model = smp.DeepLabV3(encoder_name="resnet34", encoder_weights=None, in_channels=3, classes=1)
    elif model_name == "UnetPlusPlus":
        model = smp.UnetPlusPlus(encoder_name="resnet34", encoder_weights=None, in_channels=3, classes=1)
    elif model_name == "Segformer":
        model = smp.Segformer(encoder_name="mit_b0", encoder_weights=None, in_channels=3, classes=1)
    else:
        raise ValueError(f"Unknown model: {model_name}")
    return model.to(device).eval()

def get_expected_memory_mb(model):
    param_size = sum(p.numel() * p.element_size() for p in model.parameters())
    buffer_size = sum(b.numel() * b.element_size() for b in model.buffers())
    return (param_size + buffer_size) / (1024 * 1024)

def export_to_onnx(model, dummy_input, save_path):
    torch.onnx.export(
        model,
        dummy_input,
        save_path,
        export_params=True,
        opset_version=11,
        do_constant_folding=True,
        input_names=['input'],
        output_names=['output']
    )

def benchmark_cpu_raw(model, dummy_input, runs):
    for _ in range(10):
        with torch.no_grad():
            _ = model(dummy_input)
            
    times = []
    for _ in range(runs):
        start_time = time.perf_counter()
        with torch.no_grad():
            _ = model(dummy_input)
        times.append((time.perf_counter() - start_time) * 1000)
        
    return times

def benchmark_gpu_raw(model, dummy_input, runs):
    for _ in range(10):
        with torch.no_grad():
            _ = model(dummy_input)
    torch.cuda.synchronize()
    
    times = []
    start_events = [torch.cuda.Event(enable_timing=True) for _ in range(runs)]
    end_events = [torch.cuda.Event(enable_timing=True) for _ in range(runs)]
    
    for i in range(runs):
        start_events[i].record()
        with torch.no_grad():
            _ = model(dummy_input)
        end_events[i].record()
        
    torch.cuda.synchronize()
    
    for i in range(runs):
        times.append(start_events[i].elapsed_time(end_events[i]))
        
    return times

def benchmark_onnx_raw(onnx_path, dummy_input, device_name, runs):
    providers = ['CUDAExecutionProvider', 'CPUExecutionProvider'] if device_name == "cuda" else ['CPUExecutionProvider']
    
    session_options = ort.SessionOptions()
    session_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    
    session = ort.InferenceSession(onnx_path, sess_options=session_options, providers=providers)
    input_name = session.get_inputs()[0].name
    output_name = session.get_outputs()[0].name
    
    if device_name == "cuda":
        io_binding = session.io_binding()
        io_binding.bind_input(
            name=input_name,
            device_type='cuda',
            device_id=0,
            element_type=np.float32,
            shape=tuple(dummy_input.shape),
            buffer_ptr=dummy_input.data_ptr()
        )
        io_binding.bind_output(output_name, 'cuda')
        
        for _ in range(10):
            session.run_with_iobinding(io_binding)
            
        times = []
        for _ in range(runs):
            start_time = time.perf_counter()
            session.run_with_iobinding(io_binding)
            torch.cuda.synchronize()
            times.append((time.perf_counter() - start_time) * 1000)
            
        return times
        
    else:
        dummy_input_np = dummy_input.cpu().numpy()
        for _ in range(10):
            _ = session.run(None, {input_name: dummy_input_np})
            
        times = []
        for _ in range(runs):
            start_time = time.perf_counter()
            _ = session.run(None, {input_name: dummy_input_np})
            times.append((time.perf_counter() - start_time) * 1000)
            
        return times

def run_benchmarks_and_chart():
    if "CUDAExecutionProvider" not in ort.get_available_providers():
        print("Missing onnxruntime-gpu package. ONNX will not run with CUDA.")
        
    models = ["DeepLabV3", "UnetPlusPlus", "Segformer"]
    devices = ["cpu"]
    if torch.cuda.is_available():
        devices.append("cuda")
        
    input_size = (1, 3, 512, 512)
    runs = 3
    
    print(f"{'Device':<8} | {'Format':<10} | {'Model':<15} | {'Time (ms)':<15} | {'Expected Mem (MB)'}")
    print("-" * 75)
    
    chart_data_times = {}

    for device_name in devices:
        dummy_input_pt = torch.randn(input_size, device=device_name, dtype=torch.float32)
        
        for model_name in models:
            config_key = f"{device_name.upper()}_{model_name}"
            
            model = get_model(model_name, device_name)
            expected_mem = get_expected_memory_mb(model)
            
            if device_name == "cuda":
                raw_times = benchmark_gpu_raw(model, dummy_input_pt, runs)
            else:
                raw_times = benchmark_cpu_raw(model, dummy_input_pt, runs)
                
            chart_data_times[f"PyTorch baseline_{config_key}"] = raw_times
            
            time_str = f"{np.mean(raw_times):.2f} ± {np.std(raw_times):.2f}"
            print(f"{device_name:<8} | {'PyTorch':<10} | {model_name:<15} | {time_str:<15} | {expected_mem:.2f}")
            
            onnx_path = f"{model_name}.onnx"
            if not os.path.exists(onnx_path):
                export_model = get_model(model_name, "cpu")
                export_input = torch.randn(input_size, device="cpu", dtype=torch.float32)
                export_to_onnx(export_model, export_input, onnx_path)
                del export_model
                
            onnx_raw_times = benchmark_onnx_raw(onnx_path, dummy_input_pt, device_name, runs)
            
            chart_data_times[f"ONNX optimized_{config_key}"] = onnx_raw_times
            
            onnx_time_str = f"{np.mean(onnx_raw_times):.2f} ± {np.std(onnx_raw_times):.2f}"
            print(f"{device_name:<8} | {'ONNX':<10} | {model_name:<15} | {onnx_time_str:<15} | {expected_mem:.2f}")
            
            del model
            if device_name == "cuda":
                torch.cuda.empty_cache()

    print("\nGenerating charts...")
    
    output_dir = "benchmark_charts"
    os.makedirs(output_dir, exist_ok=True)
    
    for model in models:
        labels = []
        data = []
        for config, times in chart_data_times.items():
            if model in config:
                label = config.replace(f"_{model}", "")
                labels.append(label)
                data.append(times)

        plt.figure(figsize=(10, 6))
        plt.boxplot(data, labels=labels, patch_artist=True, 
                    boxprops=dict(facecolor='lightblue', color='blue'),
                    medianprops=dict(color='red'))
        
        plt.title(f"{model} inferencijos laiko pasiskirstymas")
        plt.ylabel("Laikas (ms)")
        plt.grid(axis='y', linestyle='--', alpha=0.7)
        plt.xticks(rotation=15)
        plt.tight_layout()
        
        save_path = os.path.join(output_dir, f"boxplot_{model}.png")
        plt.savefig(save_path)
        plt.close()

    print(f"All charts saved in directory: '{output_dir}/'")

run_benchmarks_and_chart()

Device   | Format     | Model           | Time (ms)       | Expected Mem (MB)
---------------------------------------------------------------------------
cpu      | PyTorch    | DeepLabV3       | 270.21 ± 1.79   | 99.29
cpu      | ONNX       | DeepLabV3       | 218.15 ± 15.23  | 99.29
cpu      | PyTorch    | UnetPlusPlus    | 368.94 ± 16.73  | 99.56
cpu      | ONNX       | UnetPlusPlus    | 186.43 ± 3.38   | 99.56
cpu      | PyTorch    | Segformer       | 118.95 ± 0.44   | 14.17
cpu      | ONNX       | Segformer       | 62.32 ± 1.23    | 14.17
cuda     | PyTorch    | DeepLabV3       | 20.36 ± 0.09    | 99.29
cuda     | ONNX       | DeepLabV3       | 19.62 ± 0.06    | 99.29
cuda     | PyTorch    | UnetPlusPlus    | 23.62 ± 0.08    | 99.56
cuda     | ONNX       | UnetPlusPlus    | 22.76 ± 0.09    | 99.56
cuda     | PyTorch    | Segformer       | 7.25 ± 0.04     | 14.17
cuda     | ONNX       | Segformer       | 6.53 ± 0.04     | 14.17

Generating charts...


/tmp/nix-shell.fRjzOZ/ipykernel_684607/1717145743.py:190: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(data, labels=labels, patch_artist=True,
/tmp/nix-shell.fRjzOZ/ipykernel_684607/1717145743.py:190: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(data, labels=labels, patch_artist=True,
/tmp/nix-shell.fRjzOZ/ipykernel_684607/1717145743.py:190: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(data, labels=labels, patch_artist=True,


All charts saved in directory: 'benchmark_charts/'
